In [ ]:
# IMPORT AUPassata Regular + parse config

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml

font_path = './AUPassata_Rg.ttf'
fm.fontManager.addfont(font_path)


def init_plotting():
    plt.rcParams['figure.figsize'] = (8, 3)
    plt.rcParams['font.size'] = 10
    plt.rcParams['font.family'] = 'AU Passata'
    plt.rcParams['axes.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['axes.titlesize'] = 1.5 * plt.rcParams['font.size']
    plt.rcParams['legend.fontsize'] = plt.rcParams['font.size']
    plt.rcParams['xtick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['ytick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['savefig.dpi'] = 200
    plt.rcParams['xtick.major.size'] = 3
    plt.rcParams['xtick.major.width'] = 1
    plt.rcParams['ytick.major.size'] = 3
    plt.rcParams['ytick.major.width'] = 1
    plt.rcParams['legend.frameon'] = False
    plt.rcParams['axes.linewidth'] = 1

    ax = plt.gca()
    ax.spines['right'].set_color('none')
    ax.spines['top'].set_color('none')
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')


with open('../confs/thesis.yaml', 'r') as f:
    config = yaml.safe_load(f)

TRAINING_INPUT_DIR = f'../{config["data"]["input_frags_dir"]}'
INFERENCE_INPUT_DIR = f'../{config["data"]["inference_frags_dir"]}'
TRAINING_DHS_DIR = f'../{config["data"]["training_dhs_dir"]}'

TRAINING_FRAGS_DIR = f'../{config["data"]["training_base_matrices"]}'
PREPROCESSED_TRAINING_FRAGS_DIR = f'../{config["data"]["training_output_dir"]}'

TRAINING_METADATA = f'../{config["data"]["training_metadata_path"]}'
INFERENCE_METADATA = f'../{config["data"]["inference_metadata_path"]}'

COVERAGE_HANDLING = config['preprocessing']['coverage_handling']
MATRIX_COLUMNS = config['matrix']['columns']
MATRIX_ROWS = config['matrix']['rows']

COVERAGE_SUFFIX_BY_MODE = {
    'downsample': '_downsampled',
    'normalize': '_normalized',
    'none': '',
}
COVERAGE_SUFFIX = COVERAGE_SUFFIX_BY_MODE[COVERAGE_HANDLING]
COVERAGE_LABEL = {
    'downsample': 'Coverage',
    'normalize': 'CPM fraction (raw_count * 1e6 / sample_total)',
    'none': 'Coverage',
}[COVERAGE_HANDLING]

In [ ]:
# Get sample coverage - Appendix Fig 1.

import glob
import subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import numpy as np

training_files = sorted(glob.glob(f'{TRAINING_INPUT_DIR}/*/frag.tsv.gz'))
inference_files = sorted(glob.glob(f'{INFERENCE_INPUT_DIR}/*/frag.tsv.gz'))

print(training_files, inference_files)

print(f'Training files: {len(training_files)}')
print(f'Inference files: {len(inference_files)}')


def count_lines(path):
    result = subprocess.run(
        f'zcat {path} | wc -l', shell=True, capture_output=True, text=True, check=True
    )
    return int(result.stdout.strip())


def count_all(files, workers=8):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(tqdm(ex.map(count_lines, files), total=len(files)))


training_counts = count_all(training_files)
inference_counts = count_all(inference_files)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

mn, mx = float(min(training_counts)), float(max(training_counts))
avg, std = np.mean(training_counts).item(), np.std(training_counts).item()
axes[0].hist(training_counts, bins=100, edgecolor='black', alpha=0.8)
axes[0].plot([], [], ' ', label=f'Min coverage = {mn:,.2f}')
axes[0].plot([], [], ' ', label=f'Max coverage = {mx:,.2f}')
axes[0].plot([], [], ' ', label=f'Mean coverage = {avg:,.2f}')
axes[0].plot([], [], ' ', label=f'SD coverage = {std:,.2f}')
axes[0].set_ylabel('Number of samples')
axes[0].set_title(f'Training [Cristiano et al. (2019)]\nn={len(training_counts)}', fontsize=12)
axes[0].legend(loc='best')

mn, mx = float(min(inference_counts)), float(max(inference_counts))
avg, std = np.mean(inference_counts).item(), np.std(inference_counts).item()
axes[1].hist(inference_counts, bins=100, edgecolor='black', alpha=0.8)
axes[1].plot([], [], ' ', label=f'Min coverage = {mn:,.2f}')
axes[1].plot([], [], ' ', label=f'Max coverage = {mx:,.2f}')
axes[1].plot([], [], ' ', label=f'Mean coverage = {avg:,.2f}')
axes[1].plot([], [], ' ', label=f'SD coverage = {std:,.2f}')
axes[1].set_xlabel('Coverage')
axes[1].set_ylabel('Number of samples')
axes[1].set_title(f'Validation [Mathios et al. (2021)]\nn={len(inference_counts)}', fontsize=12)
axes[1].legend(loc='best')

fig.suptitle('Coverage distribution', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize cancer type distribution - Appendix Fig 2.

import pandas as pd
import matplotlib.pyplot as plt

training_metadata = pd.read_csv(TRAINING_METADATA, sep='\t')
inference_metadata = pd.read_csv(INFERENCE_METADATA, sep='\t')

train_counts = training_metadata['phenotype'].value_counts()
infer_counts = inference_metadata['Patient type'].value_counts()

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

ax = axes[0]
bars = train_counts.plot.bar(ax=ax, edgecolor='black', alpha=0.8)
ax.bar_label(ax.containers[0], padding=3)
ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
ax.tick_params(axis='x', labelrotation=45)
ax.set_title(f'Training phenotype distribution (n={train_counts.sum()})')
ax.set_xlabel('Phenotype', fontsize=14)
ax.set_ylabel('Count', fontsize=14)

ax = axes[1]
bars = infer_counts.plot.bar(ax=ax, edgecolor='black', alpha=0.8)
ax.bar_label(ax.containers[0], padding=3)
ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
ax.tick_params(axis='x', labelrotation=45)
ax.set_title(f'Inference patient type distribution (n={infer_counts.sum()})')
ax.set_xlabel('Phenotype', fontsize=14)
ax.set_ylabel('Count', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Get Lymphoid and Lymphoid_negative DHSs "coverage" (number of DHS sites) - APpendix Fig 3.

import glob
import re
from collections import defaultdict
import numpy as np

tag = f'wl{MATRIX_COLUMNS}'
initial_dhs_files = sorted(f for f in glob.glob(f'{TRAINING_DHS_DIR}*.bed') if tag not in f)
preprocessed_dhs_files = sorted(glob.glob(f'{TRAINING_DHS_DIR}*{tag}.bed'))


def clean_tag(name: str) -> str:
    return re.sub(r'(_wl\d+)?\.bed$', '', name)


groups = ['initial', 'preprocessed']
coverages = defaultdict(list)
for f1, f2 in zip(initial_dhs_files, preprocessed_dhs_files):
    tag1, tag2 = (
        clean_tag(f1.rsplit('/', 1)[1]),
        clean_tag(f2.rsplit('/', 1)[1]),
    )
    cov1 = int(subprocess.check_output(['wc', '-l', f1]).split()[0])
    coverages[tag1].append(cov1)
    cov2 = int(subprocess.check_output(['wc', '-l', f2]).split()[0])
    coverages[tag2].append(cov2)


groups, coverages
x = np.arange(len(groups))
width = 0.25
multiplier = 0

init_plotting()

for tag, coverage in coverages.items():
    offset = width * multiplier
    rects = plt.bar(x + offset, coverage, width, label=tag)
    plt.bar_label(rects, padding=3)
    multiplier += 1

plt.xlabel('Phases')
plt.ylabel('Coverage')
plt.xticks(x + (width / 2), groups)
plt.legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
# Show output matrixes coverage (initial, preprocessed, downsampled/normalized, rebinned) - Main Fig 1.

import glob
import matplotlib.pyplot as plt
import numpy as np


initial_matrices = sorted(glob.glob(f'{TRAINING_FRAGS_DIR}*.npy'))
before_rebinned_files = sorted(
    glob.glob(f'{PREPROCESSED_TRAINING_FRAGS_DIR}*{COVERAGE_SUFFIX}.npy')
)
after_rebinned_files = sorted(glob.glob(f'{PREPROCESSED_TRAINING_FRAGS_DIR}/*_rebinned.npy'))

print(f'Initial matrices files: {len(initial_matrices)}')
print(f'Before rebinned files: {len(before_rebinned_files)}')
print(f'After rebinned files: {len(after_rebinned_files)}')


def count_sum(path):
    return np.load(path).sum()


def count_all(files, workers=8):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(tqdm(ex.map(count_sum, files), total=len(files)))


initial_matrices_counts = count_all(initial_matrices)
before_rebinned_counts = count_all(before_rebinned_files)
after_rebinned_counts = count_all(after_rebinned_files)

fig, axes = plt.subplots(
    2,
    2,
    sharey=True,
    figsize=(10, 8),
)

mn, mx = min(training_counts), max(training_counts)
avg, std = np.mean(training_counts).item(), np.std(training_counts).item()
axes[0][0].hist(training_counts, bins=100, edgecolor='black', alpha=0.8)
axes[0][0].plot([], [], ' ', label=f'Min coverage = {mn:,.2f}')
axes[0][0].plot([], [], ' ', label=f'Max coverage = {mx:,.2f}')
axes[0][0].plot([], [], ' ', label=f'Mean coverage = {avg:,.2f}')
axes[0][0].plot([], [], ' ', label=f'SD coverage = {std:,.2f}')
axes[0][0].legend(loc='best')
axes[0][0].set_ylabel('Number of samples')
axes[0][0].set_xlabel('Coverage')
axes[0][0].set_title(f'Training [Cristiano et al. (2019)]\nn={len(training_counts)}', fontsize=12)

mn, mx = min(initial_matrices_counts), max(initial_matrices_counts)
avg, std = np.mean(initial_matrices_counts).item(), np.std(initial_matrices_counts).item()
axes[0][1].hist(initial_matrices_counts, bins=100, edgecolor='black', alpha=0.8)
axes[0][1].plot([], [], ' ', label=f'Min coverage = {mn:,.2f}')
axes[0][1].plot([], [], ' ', label=f'Max coverage = {mx:,.2f}')
axes[0][1].plot([], [], ' ', label=f'Mean coverage = {avg:,.2f}')
axes[0][1].plot([], [], ' ', label=f'SD coverage = {std:,.2f}')
axes[0][1].legend(loc='best')
axes[0][1].set_ylabel('Number of samples')
axes[0][1].set_xlabel('Coverage')
axes[0][1].set_title(
    f'Initial preprocessed fragments (matrices)\nn={len(initial_matrices)}', fontsize=12
)

mn, mx = min(before_rebinned_counts), max(before_rebinned_counts)
avg, std = np.mean(before_rebinned_counts).item(), np.std(before_rebinned_counts).item()
axes[1][0].hist(before_rebinned_counts, bins=100, edgecolor='black', alpha=0.8)
axes[1][0].plot([], [], ' ', label=f'Min coverage = {mn:,.2f}')
axes[1][0].plot([], [], ' ', label=f'Max coverage = {mx:,.2f}')
axes[1][0].plot([], [], ' ', label=f'Mean coverage = {avg:,.2f}')
axes[1][0].plot([], [], ' ', label=f'SD coverage = {std:,.2f}')
axes[1][0].legend(loc='best')
axes[1][0].set_ylabel('Number of samples')
axes[1][0].set_xlabel(COVERAGE_LABEL)
axes[1][0].set_title(
    f'{COVERAGE_HANDLING.title()}d matrices\nn={len(before_rebinned_counts)}', fontsize=12
)

mn, mx = min(after_rebinned_counts), max(after_rebinned_counts)
avg, std = np.mean(after_rebinned_counts).item(), np.std(after_rebinned_counts).item()
axes[1][1].hist(after_rebinned_counts, bins=100, edgecolor='black', alpha=0.8)
axes[1][1].plot([], [], ' ', label=f'Min coverage = {mn:,.2f}')
axes[1][1].plot([], [], ' ', label=f'Max coverage = {mx:,.2f}')
axes[1][1].plot([], [], ' ', label=f'Mean coverage = {avg:,.2f}')
axes[1][1].plot([], [], ' ', label=f'SD coverage = {std:,.2f}')
axes[1][1].legend(loc='best')
axes[1][1].set_ylabel('Number of samples')
axes[1][1].set_xlabel(COVERAGE_LABEL)
axes[1][1].set_title(f'Rebinned matrices\nn={len(after_rebinned_counts)}', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Show global patterns for both Lymphoid and Lymphoid_negative DHS

import glob
import numpy as np
from collections import defaultdict
from functools import partial


output_files = glob.glob(f'{PREPROCESSED_TRAINING_FRAGS_DIR}*{COVERAGE_SUFFIX}.npy')

first_matrix = np.load(output_files[0])


def clean_tag(name: str) -> str:
    m = re.search(r'__([^_.]+(?:_negative)?)\_downsampled.npy$', name)
    return m.group(1) if m else ''


sum_signal = defaultdict(partial(np.zeros, first_matrix.shape))
for f in output_files:
    matrix = np.load(f)
    tag = clean_tag(f)
    if tag:
        sum_signal[tag] += matrix


dhs_pos = first_matrix.shape[1] // 2
relative_midpoints = np.sum(sum_signal['Lymphoid'], axis=0)  # (2000,)
# relative_midpoints = relative_midpoints / relative_midpoints.sum()
frag_lengths = np.sum(sum_signal['Lymphoid'], axis=1)  # (300,)

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
fig.suptitle(
    'Summary of global patterns using downsampled and sliced fragments and both the Lymphoid and Lymphoid_negative DHSs',
    fontsize=16,
)

lengths = np.arange(len(frag_lengths))
axes[0, 0].bar(lengths, frag_lengths, width=1)
axes[0, 0].set_title('Fragment length distribution\n(Lymphoid)')
axes[0, 0].set_xlabel('Fragment length')
axes[0, 0].set_ylabel('Count')

x = np.arange(len(relative_midpoints))
axes[0, 1].plot(x, relative_midpoints)
axes[0, 1].axvline(x=dhs_pos, color='red', linestyle='--', linewidth=2, label='DHS site')
# axes[0, 1].axvline(x=dhs_pos-1000, color='green', linestyle='--', linewidth=2, label='Lower threshold')
# axes[0, 1].axvline(x=dhs_pos+1000, color='green', linestyle='--', linewidth=2, label='Upper threshold')
axes[0, 1].set_title('Relative midpoint coverage\n(Lymphoid)')
axes[0, 1].set_xlabel('Relative position')
axes[0, 1].set_ylabel('Count')

relative_midpoints = np.sum(sum_signal['Lymphoid_negative'], axis=0)  # (2000,)
# relative_midpoints = relative_midpoints / relative_midpoints.sum()
frag_lengths = np.sum(sum_signal['Lymphoid_negative'], axis=1)  # (300,)

lengths = np.arange(len(frag_lengths))
axes[1, 0].bar(lengths, frag_lengths, width=1)
axes[1, 0].set_title('Fragment length distribution\n(Lymphoid_negative)')
axes[1, 0].set_xlabel('Fragment length')
axes[1, 0].set_ylabel('Count')

x = np.arange(len(relative_midpoints))
axes[1, 1].plot(x, relative_midpoints)
axes[1, 1].axvline(x=dhs_pos, color='red', linestyle='--', linewidth=2, label='DHS site')
# axes[1, 1].axvline(x=dhs_pos-1000, color='green', linestyle='--', linewidth=2, label='Lower threshold')
# axes[1, 1].axvline(x=dhs_pos+1000, color='green', linestyle='--', linewidth=2, label='Upper threshold')
axes[1, 1].set_title('Relative midpoint coverage\n(Lymphoid_negative)')
axes[1, 1].set_xlabel('Relative position')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Investigate fragment length correlation between relative midpoints
def rolling_mean(x, window):
    return np.convolve(x, np.ones(window), mode='valid') / window


window = 25
dhs = 'Lymphoid'
frag_lengths = np.sum(sum_signal[dhs], axis=1)  # (300,)

# get median fragment length
cdf = np.cumsum(frag_lengths)
total = cdf[-1]

q10_fragment_length = np.searchsorted(cdf, 0.1 * total)
median_fragment_length = np.searchsorted(cdf, 0.50 * total)
q90_fragment_length = np.searchsorted(cdf, 0.9 * total)

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
fig.suptitle('', fontsize=16)

below_median_fl_relative_midpoints = np.sum(
    sum_signal[dhs][:median_fragment_length, :], axis=0
)  # (2000,)
# below_median_fl_relative_midpoints = below_median_fl_relative_midpoints / below_median_fl_relative_midpoints.sum()
below_median_fl_rm = rolling_mean(below_median_fl_relative_midpoints, window)

x = np.arange(len(below_median_fl_relative_midpoints))
axes[0, 0].plot(x, below_median_fl_relative_midpoints)
axes[0, 0].plot(np.arange(window, window + len(below_median_fl_rm)), below_median_fl_rm)
axes[0, 0].axvline(x=dhs_pos, color='red', linestyle='--', linewidth=2, label='DHS site')
axes[0, 0].set_title(
    f'Relative midpoint coverage\nbelow median fragment length ({median_fragment_length}bp)\n({dhs})'
)
axes[0, 0].set_xlabel('Relative position')
axes[0, 0].set_ylabel('Count')

above_median_fl_relative_midpoints = np.sum(
    sum_signal[dhs][median_fragment_length:, :], axis=0
)  # (2000,)
# above_median_fl_relative_midpoints = above_median_fl_relative_midpoints / above_median_fl_relative_midpoints.sum()
above_median_fl_rm = rolling_mean(above_median_fl_relative_midpoints, window)

x = np.arange(len(above_median_fl_relative_midpoints))
axes[1, 0].plot(x, above_median_fl_relative_midpoints)
axes[1, 0].plot(np.arange(window, window + len(above_median_fl_rm)), above_median_fl_rm)
axes[1, 0].axvline(x=dhs_pos, color='red', linestyle='--', linewidth=2, label='DHS site')
axes[1, 0].set_title(
    f'Relative midpoint coverage\nabove median fragment length ({median_fragment_length}bp)\n({dhs})'
)
axes[1, 0].set_xlabel('Relative position')
axes[1, 0].set_ylabel('Count')

q10_fl_relative_midpoints = np.sum(sum_signal[dhs][:q10_fragment_length, :], axis=0)  # (2000,)
# q10_fl_relative_midpoints = q10_fl_relative_midpoints / q10_fl_relative_midpoints.sum()
q10_fl_rm = rolling_mean(q10_fl_relative_midpoints, window)

x = np.arange(len(q10_fl_relative_midpoints))
axes[0, 1].plot(x, q10_fl_relative_midpoints)
axes[0, 1].plot(np.arange(window, window + len(q10_fl_rm)), q10_fl_rm)
axes[0, 1].axvline(x=dhs_pos, color='red', linestyle='--', linewidth=2, label='DHS site')
axes[0, 1].set_title(
    f'Relative midpoint coverage\nbelow 10% quantile fragment length ({q10_fragment_length}bp)\n({dhs})'
)
axes[0, 1].set_xlabel('Relative position')
axes[0, 1].set_ylabel('Count')

q90_fl_relative_midpoints = np.sum(sum_signal[dhs][q90_fragment_length:, :], axis=0)  # (2000,)
# q90_fl_relative_midpoints = q90_fl_relative_midpoints / q90_fl_relative_midpoints.sum()
q90_fl_rm = rolling_mean(q90_fl_relative_midpoints, window)

x = np.arange(len(q90_fl_relative_midpoints))
axes[1, 1].plot(x, q90_fl_relative_midpoints)
axes[1, 1].plot(np.arange(window, window + len(q90_fl_rm)), q90_fl_rm)
axes[1, 1].axvline(x=dhs_pos, color='red', linestyle='--', linewidth=2, label='DHS site')
axes[1, 1].set_title(
    f'Relative midpoint coverage\nabove 90% quantile fragment length ({q90_fragment_length}bp)\n({dhs})'
)
axes[1, 1].set_xlabel('Relative position')
axes[1, 1].set_ylabel('Count')


plt.tight_layout()
plt.show()

In [ ]:
# Visualize global trend for collapsing the fragment lengths
def rolling_mean(x, window):
    return np.convolve(x, np.ones(window), mode='valid') / window


window = 25

fig, axes = plt.subplots(2, 2, figsize=(14, 8))


# 1x2000
pos_relative_midpoints = np.sum(sum_signal['Lymphoid'], axis=0)
neg_relative_midpoints = np.sum(sum_signal['Lymphoid_negative'], axis=0)

pos_rm = rolling_mean(pos_relative_midpoints, window)
neg_rm = rolling_mean(neg_relative_midpoints, window)

x = np.arange(len(pos_relative_midpoints))[25:1975]

# raw signals (lighter)
axes[0, 0].plot(x, pos_relative_midpoints[25:1975], alpha=0.3, label='pos raw', color='blue')
axes[0, 0].plot(x, neg_relative_midpoints[25:1975], alpha=0.3, label='neg raw', color='orange')
# rolling averages (bold)
axes[0, 0].plot(x, pos_rm[25:1975], linewidth=2, label='pos rolling mean', color='blue')
axes[0, 0].plot(x, neg_rm[25:1975], linewidth=2, label='neg rolling mean', color='orange')
axes[0, 0].axvline(
    x=pos_relative_midpoints.shape[0] // 2,
    linestyle='--',
    linewidth=2,
    label='DHS site',
    color='red',
)
axes[0, 0].set_xlabel('Relative midpoints (25-1975)')
axes[0, 0].set_ylabel('Frags count')
axes[0, 0].legend()


# 1x800
pos_relative_midpoints = np.sum(sum_signal['Lymphoid'][130:200, 600:1400], axis=0)
neg_relative_midpoints = np.sum(sum_signal['Lymphoid_negative'][130:200, 600:1400], axis=0)

pos_rm = rolling_mean(pos_relative_midpoints, window)
neg_rm = rolling_mean(neg_relative_midpoints, window)

x = np.arange(len(pos_relative_midpoints))[25:775]

# raw signals (lighter)
axes[0, 1].plot(x, pos_relative_midpoints[25:775], alpha=0.3, label='pos raw', color='blue')
axes[0, 1].plot(x, neg_relative_midpoints[25:775], alpha=0.3, label='neg raw', color='orange')
# rolling averages (bold)
axes[0, 1].plot(x, pos_rm[25:775], linewidth=2, label='pos rolling mean', color='blue')
axes[0, 1].plot(x, neg_rm[25:775], linewidth=2, label='neg rolling mean', color='orange')
axes[0, 1].axvline(
    x=pos_relative_midpoints.shape[0] // 2,
    linestyle='--',
    linewidth=2,
    label='DHS site',
    color='red',
)
axes[0, 1].set_xlabel('Relative midpoints (25-775)')
axes[0, 1].set_ylabel('Frags count')
axes[0, 1].legend()


# 300x1
pos_fragment_lengths = np.sum(sum_signal['Lymphoid'], axis=1)
neg_fragment_lengths = np.sum(sum_signal['Lymphoid_negative'], axis=1)

x = np.arange(len(pos_fragment_lengths))

# raw signals (lighter)
axes[1, 0].plot(x, pos_fragment_lengths, label='pos raw', color='blue')
axes[1, 0].plot(x, neg_fragment_lengths, label='neg raw', color='orange')
axes[1, 0].set_xlabel('Fragment length distribution')
axes[1, 0].set_ylabel('Frags count')
axes[1, 0].legend()


# 70x1
pos_fragment_lengths = np.sum(sum_signal['Lymphoid'][130:200, 600:1400], axis=1)
neg_fragment_lengths = np.sum(sum_signal['Lymphoid_negative'][130:200, 600:1400], axis=1)

x = np.arange(len(pos_fragment_lengths))

# raw signals (lighter)
axes[1, 1].plot(x, pos_fragment_lengths, label='pos raw', color='blue')
axes[1, 1].plot(x, neg_fragment_lengths, label='neg raw', color='orange')
axes[1, 1].set_xlabel('Fragment length distribution')
axes[1, 1].set_ylabel('Frags count')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize a single sample
import numpy as np
import glob
import matplotlib.pyplot as plt

sample = 'PGDX18275P'
# sample = 'PGDX88130'
initial_matrices = glob.glob(f'{TRAINING_FRAGS_DIR}*.npy')
output_files = glob.glob(f'{PREPROCESSED_TRAINING_FRAGS_DIR}*{COVERAGE_SUFFIX}.npy')
rebinned_files = glob.glob(f'{PREPROCESSED_TRAINING_FRAGS_DIR}/*_rebinned.npy')


def get_sample_files(files):
    return [f for f in files if sample in f and 'Lymphoid' in f]


def get_sample(files, is_negative=False):
    if is_negative:
        return [f for f in files if 'negative' in f][0]
    return [f for f in files if 'negative' not in f][0]


healthy_initial_matrix = get_sample_files(initial_matrices)
healthy_output_file = get_sample_files(output_files)
healthy_rebinned = get_sample_files(rebinned_files)


neg_initial = np.load(get_sample(healthy_initial_matrix, True))
pos_initial = np.load(get_sample(healthy_initial_matrix))
neg_output = np.load(get_sample(healthy_output_file, True))
pos_output = np.load(get_sample(healthy_output_file))
neg_rebinned = np.load(get_sample(healthy_rebinned))
pos_rebinned = np.load(get_sample(healthy_rebinned))

samples = [neg_initial, pos_initial, neg_output, pos_output, neg_rebinned, pos_rebinned]

vmin = min(s.min() for s in samples)
vmax = max(s.max() for s in samples)


def plot_image(image, title, ax, vmin=vmin, vmax=vmax):
    im = ax.imshow(image, aspect='auto', origin='lower', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('Relative midpoint')
    ax.set_ylabel('Fragment length bins')
    ax.figure.colorbar(im, label='Fragment length')


fig, axes = plt.subplots(
    3,
    2,
    sharex=True,
    sharey=True,
    figsize=(14, 14),
)
plot_image(neg_initial, 'Initial negative Lymphoid DHS sample', axes[0][0])
plot_image(pos_initial, 'Initial positive Lymphoid DHS sample', axes[0][1])
plot_image(neg_output, f'{COVERAGE_HANDLING.title()}d negative Lymphoid DHS sample', axes[1][0])
plot_image(pos_output, f'{COVERAGE_HANDLING.title()}d positive Lymphoid DHS sample', axes[1][1])
plot_image(neg_rebinned, 'Rebinned negative Lymphoid DHS sample', axes[2][0])
plot_image(pos_rebinned, 'Rebinned positive Lymphoid DHS sample', axes[2][1])

In [ ]:
import glob
import numpy as np
from collections import defaultdict
from functools import partial

rebinned_matrix_paths = glob.glob(f'{PREPROCESSED_TRAINING_FRAGS_DIR}/*_rebinned.npy')
first_rebinned_matrix = np.load(rebinned_matrix_paths[0])
total_num_files = len(rebinned_matrix_paths)


def clean_tag(name: str) -> str:
    m = re.search(r'__([^_.]+(?:_negative)?)\_rebinned.npy$', name)
    return m.group(1) if m else name


sum_signal_rebinned = defaultdict(partial(np.zeros, first_rebinned_matrix.shape))
for i, f in enumerate(rebinned_matrix_paths):
    if not i % 50:
        print(f'{i}/{total_num_files}')
    rebinned_matrix = np.load(f)
    sum_signal_rebinned[clean_tag(f)] += rebinned_matrix

vmin = min(
    sum_signal_rebinned['Lymphoid'].min(),
    sum_signal_rebinned['Lymphoid_negative'].min(),
    (sum_signal_rebinned['Lymphoid'] - sum_signal_rebinned['Lymphoid_negative']).min(),
)

vmax = max(
    sum_signal_rebinned['Lymphoid'].max(),
    sum_signal_rebinned['Lymphoid_negative'].max(),
    (sum_signal_rebinned['Lymphoid'] - sum_signal_rebinned['Lymphoid_negative']).max(),
)


def plot_image(image, title, ax, vmin, vmax):
    im = ax.imshow(image, aspect='auto', origin='lower', vmin=vmin, vmax=vmax)
    # im = ax.imshow(image, aspect='auto', origin='lower')
    ax.set_title(title)
    ax.set_xlabel('Relative midpoint')
    ax.set_ylabel('Fragment length bins')
    ax.figure.colorbar(im, label='Fragment length')


fig, axes = plt.subplots(
    3,
    1,
    sharex=True,
    sharey=True,
    figsize=(14, 12),
)
plot_image(sum_signal_rebinned['Lymphoid'], 'Negative Lymphoid DHS (global)', axes[0], vmin, vmax)
plot_image(
    sum_signal_rebinned['Lymphoid_negative'], 'Positive Lymphoid DHS (global)', axes[1], vmin, vmax
)
plot_image(
    sum_signal_rebinned['Lymphoid'] - sum_signal_rebinned['Lymphoid_negative'],
    'Positive - Negative Lymphoid DHS (global)',
    axes[2],
    vmin,
    vmax,
)